# Go Beyond: Different Network Structures (D=2)

This notebook compares ER/RSC, scale-free, and small-world synthetic complexes while keeping contagion at D=2.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

CWD = Path.cwd().resolve()
candidates = [CWD, CWD.parent, CWD / 'Go_beyond', CWD.parent / 'Go_beyond']
ROOT = next(path for path in candidates if (path / 'core').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.go_beyond_experiments import run_prevalence_scan, save_pickle
from core.go_beyond_generators import generate_family_complex
from core.go_beyond_plotstyle import apply_style, ensure_output_dirs
from core.go_beyond_results import parse_results

apply_style()
FIGURES_DIR, RESULTS_DIR = ensure_output_dirs()

In [ ]:
PROFILE = 'smoke_test'  # change to 'production' for full runs

RUN_PROFILES = {
    'smoke_test': {
        'N': 220,
        't_max': 500,
        'n_simulations': 6,
        'lambda1s': np.linspace(0.2, 1.8, 18),
        'seed_pct': 5,
    },
    'production': {
        'N': 2000,
        't_max': 4000,
        'n_simulations': 80,
        'lambda1s': np.linspace(0.2, 2.2, 30),
        'seed_pct': 5,
    },
}

settings = RUN_PROFILES[PROFILE]
N = settings['N']
MU = 0.05
T_MAX = settings['t_max']
N_SIMS = settings['n_simulations']
LAMBDA1S = settings['lambda1s']
LAMBDA2_TARGET = 0.8
SEED_PCT = settings['seed_pct']

controlled_specs = {
    'er': {'k1': 16, 'k2': 4},
    'scale_free': {'k1': 16, 'k2': 4},
    'small_world': {'k1': 16, 'k2': 4},
}

native_specs = {
    'er': {'p1': 0.03, 'p2': 0.0007},
    'scale_free': {'m': 8, 'triangle_factor': 4.0},
    'small_world': {'k_nearest': 16, 'rewiring_prob': 0.12, 'triangle_factor': 4.0},
}

In [ ]:
controlled_runs = {}
for family, spec in controlled_specs.items():
    complex_data = generate_family_complex(
        family=family,
        mode='controlled',
        n_nodes=N,
        max_dimension=2,
        seed=11,
        **spec,
    )
    results = run_prevalence_scan(
        complex_data,
        lambda1s=LAMBDA1S,
        lambda2_target=LAMBDA2_TARGET,
        seed_pct=SEED_PCT,
        t_max=T_MAX,
        mu=MU,
        n_sims=N_SIMS,
    )
    controlled_runs[family] = {
        'complex_data': complex_data,
        'results': results,
        'summary': parse_results(results, cut=False),
    }

save_pickle(RESULTS_DIR / f'network_structures_controlled_{PROFILE}.pkl', controlled_runs)

In [ ]:
fig = plt.figure(figsize=(4.8, 3.4))
ax = plt.subplot(111)
styles = {
    'er': dict(marker='o', color='black', mfc='white', label='ER / RSC'),
    'scale_free': dict(marker='s', color='darkorange', mfc='white', label='Scale-free'),
    'small_world': dict(marker='^', color='cornflowerblue', mfc='white', label='Small-world'),
}

for family, payload in controlled_runs.items():
    summary = payload['summary']
    sty = styles[family].copy()
    label = sty.pop('label')
    color = sty['color']
    mfc = sty.pop('mfc')
    ax.plot(LAMBDA1S, summary['avg_rhos'], '-', linewidth=1.2, ms=6, mfc=mfc, **sty, label=label)
    ax.errorbar(LAMBDA1S, summary['avg_rhos'], yerr=summary['sem_rhos'], fmt='none', color=color, alpha=0.45, capsize=2, elinewidth=0.8)

ax.set_xlabel(r'Rescaled infectivity, $\lambda$', size=12)
ax.set_ylabel(r'Density of infected nodes, $\rho^{*}$', size=12)
ax.set_xlim(0.2, 1.8)
ax.set_ylim(-0.01, 0.85)
ax.legend(fontsize=9, handlelength=1.1, handletextpad=0.4, borderaxespad=0.2, loc='upper left')
ax.set_title(f'Controlled network comparison ({PROFILE})', fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES_DIR / f'network_structures_controlled_{PROFILE}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
native_runs = {}
for family, spec in native_specs.items():
    complex_data = generate_family_complex(
        family=family,
        mode='native',
        n_nodes=N,
        max_dimension=2,
        seed=19,
        **spec,
    )
    results = run_prevalence_scan(
        complex_data,
        lambda1s=LAMBDA1S,
        lambda2_target=LAMBDA2_TARGET,
        seed_pct=SEED_PCT,
        t_max=T_MAX,
        mu=MU,
        n_sims=N_SIMS,
    )
    native_runs[family] = {
        'complex_data': complex_data,
        'results': results,
        'summary': parse_results(results, cut=False),
    }

save_pickle(RESULTS_DIR / f'network_structures_native_{PROFILE}.pkl', native_runs)

In [ ]:
fig = plt.figure(figsize=(4.8, 3.4))
ax = plt.subplot(111)

for family, payload in native_runs.items():
    summary = payload['summary']
    sty = styles[family].copy()
    label = sty.pop('label')
    color = sty['color']
    mfc = sty.pop('mfc')
    ax.plot(LAMBDA1S, summary['avg_rhos'], '-', linewidth=1.2, ms=6, mfc=mfc, **sty, label=label)
    ax.errorbar(LAMBDA1S, summary['avg_rhos'], yerr=summary['sem_rhos'], fmt='none', color=color, alpha=0.45, capsize=2, elinewidth=0.8)

ax.set_xlabel(r'Rescaled infectivity, $\lambda$', size=12)
ax.set_ylabel(r'Density of infected nodes, $\rho^{*}$', size=12)
ax.set_xlim(0.2, 1.8)
ax.set_ylim(-0.01, 0.85)
ax.legend(fontsize=9, handlelength=1.1, handletextpad=0.4, borderaxespad=0.2, loc='upper left')
ax.set_title(f'Native-parameter comparison ({PROFILE})', fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES_DIR / f'network_structures_native_{PROFILE}.png', dpi=150, bbox_inches='tight')
plt.show()